In [ ]:
import pygame
import random

DIS_SIZE = (1280, 720)

In [ ]:
class View:
    def __init__(self):
        self.dis = pygame.display.set_mode(DIS_SIZE)
    def show(self, obj, osk = None):
        if isinstance(obj, pygame.Surface):
            self.dis.blit(obj, osk)
        elif isinstance(obj, Button):
            obj.draw()
            if isinstance(obj, Button_rect):
                pygame.draw.rect(self.dis, obj.color, (obj.x, obj.y, obj.width if obj.color==obj.move_c else obj.new_width, obj.height if obj.color==obj.move_c else obj.new_height))
            elif isinstance(obj, Button_circle):
                pygame.draw.circle(self.dis, obj.color, (obj.x, obj.y), obj.R)
            if obj.color == obj.normal_c and obj.image != None:
                self.dis.blit(obj.image, (obj.x+obj.dx if obj.dx!= None else obj.x, obj.y+obj.dy if obj.dy!= None else obj.y))
            else:
                obj.print_text()
                self.dis.blit(obj.txt, (obj.x_text, obj.y_text))
        elif isinstance(obj, Text):
            obj.print_text()
            self.dis.blit(obj.txt, (obj.x_text, obj.y_text))

In [ ]:
class System():
    def __init__(self):
        pygame.init()
        game_over = True
        
        while True:
            game = Game_Play()
            game_over = game.start()           
            
        pygame.quit()
        quit()


In [ ]:
class Text():
    def __init__(self, text_text = None, x_text = None, y_text = None, font_color=[0, 0, 0], font_size=30, font_style ="19138.ttf"):
        self.text_text=text_text
        self.x_text=x_text
        self.y_text=y_text
        self.font_color=font_color
        self.font_size=font_size
        self.font_style=font_style
        font_type = pygame.font.Font(self.font_style, self.font_size)
        self.txt = font_type.render(self.text_text, True, self.font_color)
    def print_text(self):
        font_type = pygame.font.Font(self.font_style, self.font_size)
        self.txt = font_type.render(self.text_text, True, self.font_color)

In [ ]:
class Button(Text):
    def __init__(self, x, y, normal_color, move_color):
        super().__init__()
        self.x = x
        self.y = y
        self.normal_c = normal_color
        self.move_c = move_color
        self.color = None
    def draw():
        pass

In [ ]:
class Button_rect(Button):
    def __init__(self, x, y, normal_color, move_color, width, height, text, font_size=30, action = None, image = None, act = None, new_width=None, new_height=None):
        super().__init__(x, y, normal_color, move_color)
        self.width = width
        self.height = height
        self.text = text
        self.text_text = None
        self.font_size = font_size
        self.action = action
        self.image = image
        self.act = act
        self.new_width = new_width if new_width != None else width
        self.new_height = new_height if new_height != None else height
        self.dx = 5
        self.dy = 2
    def draw(self):
        mouse = pygame.mouse.get_pos()
        isclick = pygame.mouse.get_pressed()
        if self.x < mouse[0] < self.x + self.width and self.y < mouse[1] < self.y + self.height:
            self.color = self.move_c
            self.text_text=self.text
            self.x_text=self.x+10
            self.y_text=self.y+10
            if self.action is not None and isclick[0]:
                self.action()
                pygame.time.delay(300)
        else:
            self.color = self.normal_c
            if self.image != None:
                pass
            elif self.act != None:
                self.text_text=self.act
                self.x_text=self.x+10
                self.y_text=self.y+10
            else:
                self.text_text=self.text
                self.x_text=self.x+10
                self.y_text=self.y+10

In [ ]:
class Texture():
    def __init__(self, adress):
        self.adress = adress
    def convert(self, size = None, inv = None):
        some = pygame.image.load(self.adress).convert_alpha()
        if inv == None:
            some = pygame.transform.smoothscale(some, DIS_SIZE if size==None else size)
        return some
class Music:
    def __init__(self, name):
        self.name=name
        self.repeat=1
        self.value=0.4
    def play(self):
        pygame.mixer.music.load(self.name)
        pygame.mixer.music.play(self.repeat)
        pygame.mixer.music.set_volume(self.value)

In [ ]:
class Game_Play(View):
    def __init__(self):
        
        super().__init__()
        self._game_over = False
        
        self.flag = 8
        self.flag_mode = False
        self.color_cell = 0
        
        self.game_over_text = Text("GAME OVER", 420, 320, font_size=180, font_color=[0, 0, 0])
        self.game_win_text = Text("YOU ARE AMAZING!", 275, 320, font_size=180, font_color=[0, 0, 0])
        self.main_screen = Button_rect(0, 0, [194, 179, 196], [194, 179, 196], 1280, 720, 'Game')
        self.flag_table = Button_rect(280, 10, [176, 163, 181], [176, 163, 181], 120, 60, 'Flags:' + str(self.flag))
        
        self.flag_mode_table = Button_rect(40, 300, [169, 154, 138], [132, 118, 104], 120, 80, 'Flag_mode', action = self.flag_mode_action)
        
        self.bomb = random.sample(range(64), 8)
        
    def mesh(self):
        self.cell = [[Button_rect(280+i*81, 80+j*81, [169, 154, 138], [132, 118, 104], 80, 80, '', action = self.mesh_action_create) for j in range(8)] for i in range(8)]
    def mesh_show(self):
        for i in range(8):
            for j in range(8):
                self.show(self.cell[i][j])
    def mesh_action_create(self, i = None, j = None):
        if i == None and j == None:
            i, j = pygame.mouse.get_pos()
            i = (i - 280)//81
            j = (j - 80)//81
        if self.flag_mode == True:
            if self.cell[i][j].text == "Flag":
                self.cell[i][j].text =  ""
                self.cell[i][j].normal_c =  [169, 154, 138]
                self.flag = self.flag + 1
                self.color_cell = self.color_cell - 1
            elif self.cell[i][j].text == "":
                self.cell[i][j].text =  "Flag"
                self.cell[i][j].normal_c =  [132, 118, 104]
                self.flag = self.flag - 1
                self.color_cell = self.color_cell + 1
            self.flag_table.text = 'Flags:' + str(self.flag)
        elif i+j*8 in self.bomb:
            self._game_over = True
            
            self.cell[i][j].text =  "BOMB!"
            self.cell[i][j].normal_c =  [132, 118, 104]
        elif self.cell[i][j].normal_c !=  [132, 118, 104]:
            self.cell[i][j].normal_c =  [132, 118, 104]
            summ = 0
            if i>0:
                if j>0:
                    summ = summ + ((i-1)+(j-1)*8 in self.bomb)
                
                if j<7:
                    summ = summ + ((i-1)+(j+1)*8 in self.bomb)
                summ = summ + ((i-1)+(j)*8 in self.bomb)
            if j>0:
                summ = summ+ ((i)+(j-1)*8 in self.bomb)
            if j<7:
                summ = summ + ((i)+(j+1)*8 in self.bomb)
            if i<7:
                if j>0:
                    summ = summ + ((i+1)+(j-1)*8 in self.bomb)
                
                if j<7:
                    summ = summ + ((i+1)+(j+1)*8 in self.bomb)
                summ = summ + ((i+1)+(j)*8 in self.bomb)
            if not summ:
                if i>0:
                    if j>0:
                        self.mesh_action_create((i-1), (j-1))
                    if j<7:
                        self.mesh_action_create((i-1), (j+1))
                    self.mesh_action_create((i-1), (j))
                if j>0:
                    self.mesh_action_create((i), (j-1))
                if j<7:
                    self.mesh_action_create((i), (j+1))
                if i<7:
                    if j>0:
                        self.mesh_action_create((i+1), (j-1))

                    if j<7:
                        self.mesh_action_create((i+1), (j+1))
                    self.mesh_action_create((i+1), (j))
            self.cell[i][j].text =  str(summ)
            self.color_cell = self.color_cell + 1

    def flag_mode_action(self):
        self.flag_mode = not self.flag_mode
        self.flag_mode_table = Button_rect(40, 300, [169, 154, 138] if not self.flag_mode else [132, 118, 104], [132, 118, 104], 120, 80, 'Flag_mode', action = self.flag_mode_action)
    def start(self):
        self._main_menu_over = True
        self._game_over = False
        self.mesh()
        
        clock = pygame.time.Clock()

        while not self._game_over and self.color_cell != 64:
            for event in pygame.event.get():
                if event.type==pygame.QUIT:
                    self._game_over = True
            self.show(self.main_screen)
            self.mesh_show()
            self.show(self.flag_table)
            self.show(self.flag_mode_table)
            
            pygame.display.update()
            clock.tick(20)
        if self._game_over:
            self.show(self.game_over_text)
            pygame.display.update()
            pygame.time.wait(4000)
            
        if self.color_cell == 64:
            self.show(self.game_win_text)
            Music('Victory.mp3').play()
            pygame.display.update()
            pygame.time.wait(8000)
        return self._game_over
            

System()
pass